# Gemma 4 E2B × TinyCeNN PDelta3-CLVR — Sequential Test

This copies the working Qwen3.5 sequential PDelta3/GDN2 + Local32 + CLVR experiment and adapts it to `google/gemma-4-E2B`.

The pilot targets Gemma 4's first three **pre-KV-sharing full-attention layers: 4, 9, 14**. Layer 14 preserves the KV handoff required by Gemma 4's later shared-attention layers.


In [ ]:
import os, sys, pathlib, subprocess, json
REPO_DIR = pathlib.Path('/content/TinyCeNN-LM')
if not REPO_DIR.exists():
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q','--upgrade',
    'transformers==5.17.0','datasets>=3,<5','huggingface_hub>=0.34,<2',
    'safetensors','pandas','matplotlib','-e',str(REPO_DIR)
], check=True)

preflight = (
    "import transformers; "
    "from transformers import Gemma4ForCausalLM; "
    "from transformers.models.gemma4.modeling_gemma4 import apply_rotary_pos_emb; "
    "print('transformers:', transformers.__version__); "
    "print('Gemma4 API preflight: OK')"
)
subprocess.run([sys.executable,'-c',preflight], check=True)
for p in (REPO_DIR/'src', REPO_DIR, REPO_DIR/'scripts'):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/TinyCeNN-LM')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
from huggingface_hub import get_token, login, HfApi
BASE_MODEL = 'google/gemma-4-E2B'
hf_token = get_token()
if not hf_token:
    print('Log in with a Hugging Face token that has access to Gemma 4.')
    login(add_to_git_credential=False)
    hf_token = get_token()
if not hf_token:
    raise RuntimeError('No Hugging Face token available.')
info = HfApi(token=hf_token).model_info(BASE_MODEL)
print('HF access OK:', info.id)


In [ ]:
FEATURE_DIM = 96
LOCAL_WINDOW = 32
CHUNK_SIZE = 32
CONV_KERNEL = 4
STATE_DTYPE = 'fp16'
LOCAL_GATE_INIT = 0.72
TARGET_FULL_LAYERS = 3
CONTEXT_LENGTH = 64
PROBE_CONTEXT = 64
PROBE_BLOCKS = 4
SEED = 2026
MIN_LAYER_STEPS = 50
MAX_LAYER_STEPS = 180
CHECK_EVERY = 25
LAYER_LR = 2e-4
QKV_LR_SCALE = 0.10
TEMPERATURE = 1.5
FUNCTIONAL_WEIGHT = 0.30
KL_WEIGHT = 1.00
CE_WEIGHT = 0.08
COSINE_WEIGHT = 0.20
LOCAL_GATE_PENALTY = 0.001
RESCUE_LR_SCALE = 0.50
RESCUE_FUNCTIONAL_WEIGHT = 0.15
RESCUE_KL_WEIGHT = 1.50
RESCUE_CE_WEIGHT = 0.12
ACCEPT_NMSE = 0.15
ACCEPT_COSINE = 0.94
ACCEPT_INCREMENTAL_DELTA_NLL = 0.015
ACCEPT_CUMULATIVE_DELTA_NLL = 0.05
RESUME = True
MAX_ROUNDS_PER_RUN = 2
MAX_RUNTIME_MINUTES = 240
OUTPUT_DIR = DRIVE_ROOT / f'gemma4-e2b-pdelta3-gdn2-clvr-local{LOCAL_WINDOW}-f{FEATURE_DIM}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT_DIR)


## Train / resume

This uses the same acceptance gates and checkpoint logic as the working Qwen3.5 test. The child log is streamed live and saved to Google Drive.


In [ ]:
import signal
subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only'], check=True)
cmd = [
    sys.executable,'-u',str(REPO_DIR/'scripts'/'run_gemma4_e2b_pdelta3_clvr.py'),
    '--base-model',BASE_MODEL,'--output-dir',str(OUTPUT_DIR),
    '--feature-dim',str(FEATURE_DIM),'--local-window',str(LOCAL_WINDOW),
    '--chunk-size',str(CHUNK_SIZE),'--conv-kernel',str(CONV_KERNEL),
    '--state-dtype',STATE_DTYPE,'--local-gate-init',str(LOCAL_GATE_INIT),
    '--target-full-layers',str(TARGET_FULL_LAYERS),'--context-length',str(CONTEXT_LENGTH),
    '--probe-context',str(PROBE_CONTEXT),'--probe-blocks',str(PROBE_BLOCKS),
    '--seed',str(SEED),'--min-layer-steps',str(MIN_LAYER_STEPS),
    '--max-layer-steps',str(MAX_LAYER_STEPS),'--check-every',str(CHECK_EVERY),
    '--layer-lr',str(LAYER_LR),'--qkv-lr-scale',str(QKV_LR_SCALE),
    '--temperature',str(TEMPERATURE),'--functional-weight',str(FUNCTIONAL_WEIGHT),
    '--kl-weight',str(KL_WEIGHT),'--ce-weight',str(CE_WEIGHT),
    '--cosine-weight',str(COSINE_WEIGHT),'--local-gate-penalty',str(LOCAL_GATE_PENALTY),
    '--rescue-lr-scale',str(RESCUE_LR_SCALE),'--rescue-functional-weight',str(RESCUE_FUNCTIONAL_WEIGHT),
    '--rescue-kl-weight',str(RESCUE_KL_WEIGHT),'--rescue-ce-weight',str(RESCUE_CE_WEIGHT),
    '--accept-nmse',str(ACCEPT_NMSE),'--accept-cosine',str(ACCEPT_COSINE),
    '--accept-incremental-delta-nll',str(ACCEPT_INCREMENTAL_DELTA_NLL),
    '--accept-cumulative-delta-nll',str(ACCEPT_CUMULATIVE_DELTA_NLL),
    '--max-runtime-minutes',str(MAX_RUNTIME_MINUTES),
    '--warm-start-previous-core','--train-qkv','--strict-acceptance',
    '--resume' if RESUME else '--no-resume'
]
env = os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['SEQUENTIAL_MAX_ROUNDS_PER_RUN']=str(MAX_ROUNDS_PER_RUN); env['HF_TOKEN']=hf_token
log_path = OUTPUT_DIR/'last_gemma4_colab_run.log'
print(' '.join(cmd)); print('Log:', log_path)
interrupted=False
with log_path.open('w',encoding='utf-8') as log:
    p = subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    assert p.stdout is not None
    try:
        for line in p.stdout:
            print(line,end=''); log.write(line); log.flush()
        rc=p.wait()
    except KeyboardInterrupt:
        interrupted=True; p.send_signal(signal.SIGINT)
        try: rc=p.wait(timeout=30)
        except subprocess.TimeoutExpired: p.terminate(); rc=p.wait()
print('Process return code:',rc)
if rc != 0 and not interrupted:
    lines=log_path.read_text(encoding='utf-8',errors='replace').splitlines()
    print('\n--- LAST 100 LOG LINES ---'); print('\n'.join(lines[-100:]))
    raise RuntimeError(f'Gemma4 trainer failed with return code {rc}. See {log_path}.')


In [ ]:
import pandas as pd
for name in ('qwen35_run_status.json','qwen35_progress.json','qwen35_in_progress.json'):
    p=OUTPUT_DIR/name
    if p.exists():
        print('\n###',name); print(p.read_text()[:14000])
p=OUTPUT_DIR/'qwen35_progress.json'
if p.exists():
    reports=json.loads(p.read_text()).get('reports',[])
    if reports:
        df=pd.DataFrame(reports)
        cols=[c for c in ['layer','round','step','accepted','nmse','cosine','incremental_delta_nll','cumulative_delta_nll','local_gate_mean'] if c in df.columns]
        display(df[cols].tail(30))


## Original Gemma 4 vs accepted PDelta3-CLVR — sample prompts

This comparison loads the two models one at a time to keep GPU memory lower.


In [ ]:
import gc
from transformers import AutoTokenizer, Gemma4ForCausalLM
from run_gemma4_e2b_pdelta3_clvr import Gemma4PDelta3CLVRConfig, replace_layers

progress_path=OUTPUT_DIR/'qwen35_progress.pt'
if not progress_path.exists(): raise FileNotFoundError('No accepted checkpoint yet: '+str(progress_path))
progress=torch.load(progress_path,map_location='cpu',weights_only=False)
accepted=[int(x) for x in progress.get('accepted_full_attention_layers',[])]
if not accepted: raise RuntimeError('No accepted Gemma4 replacement layers yet.')
cfg=Gemma4PDelta3CLVRConfig.from_dict(progress['config'])
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype=torch.bfloat16 if device.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if device.type=='cuda' else torch.float32)
tok=AutoTokenizer.from_pretrained(BASE_MODEL,token=hf_token,use_fast=True)
if tok.pad_token_id is None: tok.pad_token=tok.eos_token
prompts=['The future of efficient language models is','A recurrent state can help a language model by','Vienna is the capital of Austria and','Explain neural networks in simple words:']

@torch.no_grad()
def run_prompts(model):
    model.eval(); out={}
    for prompt in prompts:
        x=tok(prompt,return_tensors='pt').to(device)
        y=model.generate(**x,max_new_tokens=64,do_sample=False,use_cache=False,pad_token_id=tok.eos_token_id,eos_token_id=tok.eos_token_id)
        out[prompt]=tok.decode(y[0],skip_special_tokens=True)
    return out

print('Loading original Gemma4...')
base=Gemma4ForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,token=hf_token,attn_implementation='eager').to(device); base.config.use_cache=False
base_text=run_prompts(base); del base; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

print('Loading converted Gemma4...')
new=Gemma4ForCausalLM.from_pretrained(BASE_MODEL,dtype=dtype,token=hf_token,attn_implementation='eager').to(device); new.config.use_cache=False
replace_layers(new,cfg,accepted)
inc=new.load_state_dict(progress['attention_state'],strict=False)
prefixes=tuple(f'model.layers.{i}.self_attn.' for i in accepted)
missing=[k for k in inc.missing_keys if k.startswith(prefixes)]
if missing: raise RuntimeError('Missing replacement weights: '+str(missing[:12]))
new_text=run_prompts(new)
print('Accepted layers:',accepted)
for p in prompts:
    print('\n'+'='*100); print('PROMPT:',p); print('\nBASE GEMMA4:\n'+base_text[p]); print('\nPDELTA3-CLVR:\n'+new_text[p])
